In [25]:
import os, sqlite3

In [26]:
DB_PATH = "experiment.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

setup = sqlite3.connect(DB_PATH, autocommit=False)
setup.execute("""
    CREATE TABLE clients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        CONSTRAINT uq_client_name UNIQUE (name)
    )
""")
setup.execute("""
    CREATE TABLE client_notes(
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        client_id   INTEGER NOT NULL,
        note        TEXT NOT NULL,
        CONSTRAINT fk_client_notes_client
            FOREIGN KEY (client_id) REFERENCES clients(id) ON DELETE RESTRICT
    )
""")

setup.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A",))
setup.execute("INSERT INTO clients (name) VALUES (?)", ("Farm B",))
setup.execute("INSERT INTO client_notes (client_id, note) VALUES (?, ?)", (1, "Seeding Plan"))
setup.commit()
setup.close()

def fk_on(conn):
    return conn.execute("PRAGMA foreign_keys").fetchone()[0]    # 1 = enforced, 0 = not

def orphans(conn):
    return conn.execute("PRAGMA foreign_key_check").fetchall()  # rows that violate an FK right now

def show(conn, label):
    clients = conn.execute("SELECT id, name FROM clients ORDER BY id").fetchall()
    notes = conn.execute("SELECT id, client_id, note FROM client_notes ORDER BY id").fetchall()
    print(f"{label}\n   clients: {clients}\n    notes: {notes}\n    violations: {orphans(conn)}")

## not enforcing FK

In [27]:
conn = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)
print("in_transaction: ", conn.in_transaction)
print("foreign_keys: ", fk_on(conn))

in_transaction:  True
foreign_keys:  0


In [28]:
try:
    conn.execute("DELETE FROM clients WHERE id = ?", (1,))
    print("delete succeeded")
except sqlite3.Error as err:
    print(type(err).__name__, "|", err, "|", getattr(err, "sqlite_errorname", None), "|", getattr(err, "sqlite_errorcode", None))
show(conn, "after delete attempt")
conn.rollback() # keep the data intact for the next round of experimentation

delete succeeded
after delete attempt
   clients: [(2, 'Farm B')]
    notes: [(1, 1, 'Seeding Plan')]
    violations: [('client_notes', 1, 'clients', 0)]


## with enforcing FK

### setting got ignored

In [29]:
print("in_transaction before PRAGMA:", conn.in_transaction)
conn.execute("PRAGMA foreign_keys = ON")
print("foreign_keys after setting it: ", fk_on(conn))
conn.close()

in_transaction before PRAGMA: True
foreign_keys after setting it:  0


### enable it before any transaction exists

In [30]:
enforced = sqlite3.connect(DB_PATH, autocommit=True, timeout=1)
enforced.execute("PRAGMA foreign_keys = ON")
print("foreign_keys (autocommit=True):", fk_on(enforced))

enforced.autocommit = False 
print("foreign_keys after switching to autocommit=False:", fk_on(enforced))
print("in_transaction:", enforced.in_transaction)

foreign_keys (autocommit=True): 1
foreign_keys after switching to autocommit=False: 1
in_transaction: True


In [31]:
try:
    enforced.execute("DELETE FROM clients WHERE id = ?", (1,))
    print("delete succeeded")
except sqlite3.Error as err:
    print(type(err).__name__, 
          "\n| message: ", err, 
          "\n| name:", getattr(err, "sqlite_errorname", None), 
          "\n| code", getattr(err, "sqlite_errorcode", None))
enforced.rollback()

IntegrityError 
| message:  FOREIGN KEY constraint failed 
| name: SQLITE_CONSTRAINT_TRIGGER 
| code 1811


In [32]:
try:
    enforced.execute("INSERT INTO client_notes (client_id, note) VALUES (?, ?)", (999, "Orphan note"))
    print("insert succeeded")
except sqlite3.Error as err:
    print("type: ", type(err).__name__, 
          "\n| message: ", err, 
          "\n| errorname: ", getattr(err, "sqlite_errorname", None),
          "\n| errorcode: ", getattr(err, "sqlite_errorcode", None))
enforced.rollback()

type:  IntegrityError 
| message:  FOREIGN KEY constraint failed 
| errorname:  SQLITE_CONSTRAINT_FOREIGNKEY 
| errorcode:  787


In [33]:
enforced.execute("DELETE FROM clients WHERE id = ?", (2,))
show(enforced, "after deleting Farm B (uncommitted)")
enforced.rollback()

after deleting Farm B (uncommitted)
   clients: [(1, 'Farm A')]
    notes: [(1, 1, 'Seeding Plan')]
    violations: []


In [34]:
careless = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)    # no PRAGMA
careless.execute("DELETE FROM clients WHERE id = ?", (1,))
careless.commit()
careless.close()

show(enforced, "enforced conection, after careless delete")
enforced.rollback()

enforced conection, after careless delete
   clients: [(2, 'Farm B')]
    notes: [(1, 1, 'Seeding Plan')]
    violations: [('client_notes', 1, 'clients', 0)]


In [35]:
enforced.close()